[![](imagens/colab-badge.png){width="16%"}](https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/cap02/cap02.EPs_aluno.ipynb)
[![](imagens/github-badge.png){width="20%"}](https://github.com/fzampirolli/pdi-vc)

## 💻 **Parte Prática com Exercícios de Programação**

---

### 🎯 Objetivo deste Caderno {.unnumbered}

O caderno permite desenvolver, validar, organizar e testar soluções de  **Exercícios de Programação (EPs)** em ambientes interativos, como o Colab, com os mesmos casos de teste do Moodle, copiando para lá apenas na hora de registrar a nota oficial.

#### Download {.unnumbered}

Baixe `morph.py` e `testsuite.py` executando a célula abaixo:

In [2]:
import os, sys, importlib, inspect, urllib.request

# URLs do repositório
BASE_URL = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph"
for f in ["morph.py", "testsuite.py"]:
    if not os.path.exists(f):
        urllib.request.urlretrieve(f"{BASE_URL}/{f}", f)

import morph, testsuite
importlib.reload(morph); importlib.reload(testsuite)
from morph import mm
from testsuite import TestSuite

print(f"✅ Ambiente pronto. Morph: {morph.__version__} | TestSuite: {testsuite.__version__}")

✅ Ambiente pronto. Morph: 1.1.0 | TestSuite: 1.1.0


---

#### Executando os Testes {.unnumbered}

Para rodar os testes, execute `TestSuite("EP01_01.extensão").run()` numa nova célula, trocando a extensão pela da linguagem usada (`.py`, `.java`, `.c`, `.cpp`, `.js` ou `.r`). O sistema baixa os casos de teste do GitHub, executa o programa e calcula a nota automaticamente.

---

### EP02_01 ☀️ Ajuste de Brilho e Contraste

Nesta atividade, você deve implementar uma transformação linear de intensidade para ajustar o brilho e o contraste de uma imagem.

* Leia dois inteiros **L** e **C**, representando as dimensões da matriz.
* Leia um valor real **$\alpha$** (contraste) e um inteiro **$\beta$** (brilho).
* Leia os valores inteiros da matriz.
* Para cada pixel $p$, calcule o novo valor $p'$ usando a fórmula:

$$p' = \text{clip}(\text{round}(\alpha \times p + \beta))$$

* Imprima a matriz resultante com os valores finais.

📌 **Importante**:

* **Saturação (Clip):** O resultado final deve ser obrigatoriamente um inteiro no intervalo $[0, 255]$. Valores menores que 0 tornam-se 0, e valores maiores que 255 tornam-se 255.
* **Arredondamento:** Realize o arredondamento matemático antes de converter para inteiro e aplicar o clip.
* Ver um simulador interativo para esta questão na **@fig-02-brilho-contraste** (manipule os parâmetros $\alpha$ e $\beta$ para visualizar o efeito de realce e correção).

---

#### 🧠 Transformação Linear de Intensidade

Ajustar o brilho e o contraste são operações fundamentais em PDI. Elas permitem corrigir imagens subexpostas (escuras demais) ou melhorar a visibilidade de detalhes sutis:

| Parâmetro | Função | Efeito |
| --- | --- | --- |
| **$\alpha$ (Alpha)** | Multiplicador | Ajusta o **Contraste**. $>1$ aumenta a diferença entre tons; $<1$ diminui. |
| **$\beta$ (Beta)** | Aditivo | Ajusta o **Brilho**. Valores positivos clareiam; negativos escurecem. |
| **Clip** | Limitador | Garante que o pixel permaneça dentro do padrão de 8 bits ($0-255$). |

---

#### 📋 Tarefa (especificação para VPL)

**Entrada:**

A primeira linha contém `L`.

A segunda linha contém `C`.

A terceira linha contém `alpha` ($\alpha$) e `beta` ($\beta$).

As linhas seguintes contêm os elementos da matriz.

**Saída:**

A matriz transformada com `L` linhas e `C` colunas.

---

#### 📌 Exemplos

| Entrada | Saída | Observação |
| --- | --- | --- |
| 1<br>4<br>1.5 -30<br>0 100 180 255 | 0 120 240 255 | Note o efeito de saturação no último pixel |


**Exemplo de teste de `sqrt` em Python, com `timeit` isolando cada operação:**

In [3]:
#| label: fig-02-brilho-contraste
#| fig-cap: "Simulador: Ajuste de Brilho e Contraste"
#| echo: false
#| output: true

from IPython.display import HTML
HTML("""
<div id="sim-adj-08" style="background-color: #fef9ef; border-radius: 18px; border: 1px solid #ede6d8; overflow: hidden; margin-top: 20px; font-family: sans-serif;">
    <div style="background: #f3efe6; padding: 8px 16px; font-size: 12px; color: #5e5a4a; border-bottom: 1px solid #e9dfcf; display: flex; justify-content: space-between; align-items: center;">
      <span>🎮 Simulador: Brilho e Contraste</span>
      <span style="background: #e8e0cf; border-radius: 40px; padding: 2px 10px; font-weight: 600; font-size: 10px;">☀️ p' = αp + β</span>
    </div>

    <div style="padding: 20px; background: white;">
        <div style="background: #fafafa; border: 1px solid #ddd; border-radius: 12px; padding: 20px; margin-bottom: 20px; display: grid; grid-template-columns: 1fr 1fr; gap: 30px;">
            <div>
                <div style="display: flex; justify-content: space-between; margin-bottom: 8px;">
                    <label style="font-size: 12px; font-weight: bold; color: #f39c12;">α (Contraste)</label>
                    <span id="vl-alpha-08" style="font-family: monospace; font-weight: bold; color: #f39c12;">1.0</span>
                </div>
                <input id="sl-alpha-08" style="width: 100%; accent-color: #f39c12;" max="3" min="0" step="0.1" type="range" value="1">
            </div>
            <div>
                <div style="display: flex; justify-content: space-between; margin-bottom: 8px;">
                    <label style="font-size: 12px; font-weight: bold; color: #3498db;">β (Brilho)</label>
                    <span id="vl-beta-08" style="font-family: monospace; font-weight: bold; color: #3498db;">0</span>
                </div>
                <input id="sl-beta-08" style="width: 100%; accent-color: #3498db;" max="100" min="-100" step="1" type="range" value="0">
            </div>
        </div>

        <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 20px;">
            <div style="text-align: center; background: white; border: 1px solid #eee; padding: 15px; border-radius: 12px;">
                <p style="font-size: 10px; font-weight: 600; color: #888; text-transform: uppercase; margin-bottom: 12px;">Entrada Original</p>
                <div id="grid-orig-08" style="display: grid; grid-template-columns: repeat(4, 42px); gap: 4px; justify-content: center;"></div>
                <button id="btn-new-08" style="margin-top: 15px; font-size: 11px; padding: 5px 10px; border-radius: 4px; border: 1px solid #ccc; background: white; cursor: pointer;">🎲 Nova Imagem</button>
            </div>
            
            <div style="text-align: center; background: white; border: 1px solid #eee; padding: 15px; border-radius: 12px;">
                <p style="font-size: 10px; font-weight: 600; color: #888; text-transform: uppercase; margin-bottom: 12px;">Resultado (p')</p>
                <div id="grid-new-08" style="display: grid; grid-template-columns: repeat(4, 42px); gap: 4px; justify-content: center;"></div>
                <button id="btn-reset-08" style="margin-top: 15px; font-size: 11px; padding: 5px 10px; border-radius: 4px; border: 1px solid #ccc; background: white; cursor: pointer;">↩ Resetar</button>
            </div>
        </div>

        <div id="debug-08" style="margin-top: 20px; background: #e8f5e9; border-radius: 8px; padding: 10px; border: 1px solid #c8e6c9; font-family: monospace; font-size: 11px; color: #2e7d32; text-align: center;">
            Fórmula: clip(1.0 * p + 0)
        </div>
    </div>
</div>

<script>
setTimeout(function() {
    const slAlpha = document.getElementById('sl-alpha-08');
    const slBeta = document.getElementById('sl-beta-08');
    const vlAlpha = document.getElementById('vl-alpha-08');
    const vlBeta = document.getElementById('vl-beta-08');
    const gridOrig = document.getElementById('grid-orig-08');
    const gridNew = document.getElementById('grid-new-08');
    const debug = document.getElementById('debug-08');

    if (!slAlpha || !gridOrig) return;

    let pixels = [];
    const count = 16;

    function generate() {
        pixels = Array.from({length: count}, () => Math.floor(Math.random() * 256));
    }

    function render() {
        const a = parseFloat(slAlpha.value);
        const b = parseInt(slBeta.value);

        vlAlpha.innerText = a.toFixed(1);
        vlBeta.innerText = b;
        debug.innerHTML = `Fórmula aplicada: <b>clip( round(${a.toFixed(1)} * p + (${b})) )</b>`;

        gridOrig.innerHTML = '';
        gridNew.innerHTML = '';

        pixels.forEach(p => {
            // Original
            const cellO = document.createElement('div');
            cellO.style.cssText = `width:42px; height:42px; display:flex; align-items:center; justify-content:center; font-size:10px; font-weight:bold; border-radius:4px; border:1px solid #eee;`;
            cellO.style.background = `rgb(${p},${p},${p})`;
            cellO.style.color = p > 128 ? 'black' : 'white';
            cellO.innerText = p;
            gridOrig.appendChild(cellO);

            // Transformado
            let res = Math.round(a * p + b);
            res = Math.max(0, Math.min(255, res));

            const cellN = document.createElement('div');
            cellN.style.cssText = `width:42px; height:42px; display:flex; align-items:center; justify-content:center; font-size:10px; font-weight:bold; border-radius:4px; border:1px solid #ddd; transition: 0.2s;`;
            cellN.style.background = `rgb(${res},${res},${res})`;
            cellN.style.color = res > 128 ? 'black' : 'white';
            cellN.innerText = res;
            gridNew.appendChild(cellN);
        });
    }

    slAlpha.oninput = render;
    slBeta.oninput = render;
    document.getElementById('btn-new-08').onclick = () => { generate(); render(); };
    document.getElementById('btn-reset-08').onclick = () => { slAlpha.value = 1; slBeta.value = 0; render(); };

    generate();
    render();
}, 100);
</script>
""")

In [ ]:
%%writefile EP02_01.py
# Código Python

In [ ]:
TestSuite("EP02_01.py").run()